In [6]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [7]:
df = pd.read_csv('../../tweets-data/bitcoin-2.csv')
df

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
0,1895240223195570561,Thu Feb 27 22:30:55 +0000 2025,6,Chale Your man start dey do Crypto ? Football ...,1895240223195570561,NaN,NaN,in,"Kumasi, Ghana",0,2,3,https://x.com/KingLevels6/status/1895240223195...,1371608363109388288,KingLevels6
1,1895198298522427669,Thu Feb 27 19:44:20 +0000 2025,12,Bitcoin Orange @OGsatoshis ️️️ https://t.co/79...,1895198298522427669,https://pbs.twimg.com/media/Gk0YzyRWAAA2q4Q.png,NaN,in,NaN,0,3,2,https://x.com/ajk9073/status/1895198298522427669,1643333155275829248,ajk9073
2,1894974680634925162,Thu Feb 27 18:30:27 +0000 2025,87,@WahyuS002 Problem terbesar yg diselesaikan we...,1895179706108125324,NaN,WahyuS002,in,Arbitrum,1,2,4,https://x.com/yanzero_/status/1895179706108125324,14119874,yanzero_
3,1895071930346283197,Thu Feb 27 11:22:11 +0000 2025,77,Buat ngejelasin ke boomer yang bilang Underwea...,1895071930346283197,https://pbs.twimg.com/ext_tw_video_thumb/18950...,NaN,in,Indonesia,0,2,4,https://x.com/mkbijaksana/status/1895071930346...,253610602,mkbijaksana
4,1895067504013459669,Thu Feb 27 11:04:36 +0000 2025,12,Ek aisi wife to main bhi deserve karta hu yaar...,1895067504013459669,https://pbs.twimg.com/ext_tw_video_thumb/18950...,NaN,in,India,0,3,7,https://x.com/mukesh1yadav87/status/1895067504...,1604710766854438912,mukesh1yadav87
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1235,1859162476304359719,Wed Nov 20 09:10:50 +0000 2024,269,Siapa yang akan Menjadi Raja Mata Uang Crypto...,1859162476304359719,https://pbs.twimg.com/media/Gc0SWRdbAAAkgGg.jpg,NaN,in,Indonesia,0,21,35,https://x.com/MyYunieyusuf519/status/185916247...,1547505451587506176,MyYunieyusuf519
1236,1859155836582445113,Wed Nov 20 08:44:27 +0000 2024,84,Ktk Kampuni zenye Market Cap kubwa Duniani or ...,1859155836582445113,https://pbs.twimg.com/media/Gc0Fg6pbcAAepnV.jpg,NaN,in,NaN,1,3,18,https://x.com/godbless_lema/status/18591558365...,845272399381778432,godbless_lema
1237,1859137762672423072,Wed Nov 20 07:41:58 +0000 2024,77,@meinmokhtar Hang keje tak? Penuhkan ASB(200k ...,1859140112711643172,NaN,meinmokhtar,in,Kuala Lumpur,0,3,12,https://x.com/ymandikadewa/status/185914011271...,1251404145648472071,ymandikadewa
1238,1859131096794894631,Wed Nov 20 07:06:09 +0000 2024,11,𝘽𝙞𝙩𝙘𝙤𝙞𝙣 𝙫𝙨 𝘼𝙡𝙩𝙘𝙤𝙞𝙣 Join disini: https://t.co/3...,1859131096794894631,https://pbs.twimg.com/media/GczxkG_bcAA4CW7.jpg,NaN,in,Indonesia,2,1,2,https://x.com/cryptorize_id/status/18591310967...,1815108051642593282,cryptorize_id


In [8]:
df.drop(['conversation_id_str', 'created_at', 'favorite_count', 'id_str', 'image_url', 'in_reply_to_screen_name', 'lang', 'location', 'quote_count', 'reply_count', 'retweet_count', 'tweet_url', 'user_id_str', 'username'], axis=1, inplace=True)
df.drop_duplicates(subset=['full_text'], keep='first', inplace=True)
df.info()
df

<class 'pandas.DataFrame'>
Index: 1236 entries, 0 to 1239
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   full_text  1236 non-null   str  
dtypes: str(1)
memory usage: 19.3 KB


,full_text
0,Chale Your man start dey do Crypto ? Football ...
1,Bitcoin Orange @OGsatoshis ️️️ https://t.co/79...
2,@WahyuS002 Problem terbesar yg diselesaikan we...
3,Buat ngejelasin ke boomer yang bilang Underwea...
4,Ek aisi wife to main bhi deserve karta hu yaar...
...,...
1235,Siapa yang akan Menjadi Raja Mata Uang Crypto...
1236,Ktk Kampuni zenye Market Cap kubwa Duniani or ...
1237,@meinmokhtar Hang keje tak? Penuhkan ASB(200k ...
1238,𝘽𝙞𝙩𝙘𝙤𝙞𝙣 𝙫𝙨 𝘼𝙡𝙩𝙘𝙤𝙞𝙣 Join disini: https://t.co/3...


In [9]:
#Cleaning Text
def clean_text(full_text):
    #menghapus RT tag
    t1 =re.sub('RT\\s', '', full_text)
    #menghapus @_username
    #t2 =re.sub('\B@\w', '', t1)
    t2 =re.sub(r'\@([\\w]+)',' ', t1)
    #mengganti emoji decode dengan spasi
    #t3 =emoji.demojize(t2)
    t3 = re.sub(r'\\u[a-zA-Z0-9]{4}', ' ', t2)
    #mengganti enter /n/ dengan spasi
    t4 =re.sub('\n\\s', ' ', t3)
    t5 =re.sub('\n', ' ', t4)
    #Non-ascii
    t6 = re.sub(r'[^\x00-\x7F]+',' ', t5)
    #koreksi duplikasi tiga karakter beruntun atau lebih (contoh. yukkk)
    t7 = re.sub(r'([a-zA-Z])\1\1','\\1', t6)
    #replace URL 
    t8 = re.sub(r'http[s]?\:\/\/.[a-zA-Z0-9\.\/\_?=%&#\-\+!]+',' ', t7)
    t9 = re.sub(r'pic.twitter.com?.[a-zA-Z0-9\.\/\_?=%&#\-\+!]+',' ', t8)
    #hapus tagar
    #t13 = re.sub(r'\#([\w]+)',' ', t12)
    t10 = re.sub(r'\#[a-zA-Z0-9_]+','', t9)
    #hapus angka
    t11 = re.sub(r'[0-9]+',' ', t10)
    # hapus simbol
    t12 = re.sub(r'[!$%^&*@#()_+|~=`{}\[\]%\-:";\'<>?,.\/]', ' ', t11)
    # Tahap-8: spasi ganda (atau lebih) menjadi satu spasi
    t13 = re.sub(' +', ' ', t12)
    #Tahap-9: spasi di awal dan akhir kalimat
    t14 = re.sub(r'^[ ]|[ ]$','', t13)
    return t14

In [10]:
#Apply cleaning function to the dataframe
for i in df.iterrows():
    df['full_text'] = df['full_text'].apply(clean_text)
    
df

,full_text
0,Chale Your man start dey do Crypto Football ay...
1,Bitcoin Orange OGsatoshis
2,WahyuS Problem terbesar yg diselesaikan web ad...
3,Buat ngejelasin ke boomer yang bilang Underwea...
4,Ek aisi wife to main bhi deserve karta hu yaar
...,...
1235,Siapa yang akan Menjadi Raja Mata Uang Crypto ...
1236,Ktk Kampuni zenye Market Cap kubwa Duniani or ...
1237,meinmokhtar Hang keje tak Penuhkan ASB k sbb d...
1238,Join disini Gabung belajar santuy bareng Crypt...


In [11]:
sample = 'Buat ngejelasin ke boomer yang bilang Underwear eh underlyingnya bitcoin itu apa? https://t.co/QXI68L7vfJ'
sample = clean_text(sample)
sample


'Buat ngejelasin ke boomer yang bilang Underwear eh underlyingnya bitcoin itu apa'

In [12]:
#Case Folding
df['case_folding'] = df['full_text'].str.lower()
df

,full_text,case_folding
0,Chale Your man start dey do Crypto Football ay...,chale your man start dey do crypto football ay...
1,Bitcoin Orange OGsatoshis,bitcoin orange ogsatoshis
2,WahyuS Problem terbesar yg diselesaikan web ad...,wahyus problem terbesar yg diselesaikan web ad...
3,Buat ngejelasin ke boomer yang bilang Underwea...,buat ngejelasin ke boomer yang bilang underwea...
4,Ek aisi wife to main bhi deserve karta hu yaar,ek aisi wife to main bhi deserve karta hu yaar
...,...,...
1235,Siapa yang akan Menjadi Raja Mata Uang Crypto ...,siapa yang akan menjadi raja mata uang crypto ...
1236,Ktk Kampuni zenye Market Cap kubwa Duniani or ...,ktk kampuni zenye market cap kubwa duniani or ...
1237,meinmokhtar Hang keje tak Penuhkan ASB k sbb d...,meinmokhtar hang keje tak penuhkan asb k sbb d...
1238,Join disini Gabung belajar santuy bareng Crypt...,join disini gabung belajar santuy bareng crypt...


In [13]:
print("Sebelim case folding : ", sample)

sample = sample.lower()
print("Setelah case folding : ", sample)

Sebelim case folding :  Buat ngejelasin ke boomer yang bilang Underwear eh underlyingnya bitcoin itu apa
Setelah case folding :  buat ngejelasin ke boomer yang bilang underwear eh underlyingnya bitcoin itu apa


In [14]:
#Tokenizing
tokenizer = RegexpTokenizer(r'\w+')
df['tokens'] = df['case_folding'].apply(tokenizer.tokenize)
df

,full_text,case_folding,tokens
0,Chale Your man start dey do Crypto Football ay...,chale your man start dey do crypto football ay...,"[chale, your, man, start, dey, do, crypto, foo..."
1,Bitcoin Orange OGsatoshis,bitcoin orange ogsatoshis,"[bitcoin, orange, ogsatoshis]"
2,WahyuS Problem terbesar yg diselesaikan web ad...,wahyus problem terbesar yg diselesaikan web ad...,"[wahyus, problem, terbesar, yg, diselesaikan, ..."
3,Buat ngejelasin ke boomer yang bilang Underwea...,buat ngejelasin ke boomer yang bilang underwea...,"[buat, ngejelasin, ke, boomer, yang, bilang, u..."
4,Ek aisi wife to main bhi deserve karta hu yaar,ek aisi wife to main bhi deserve karta hu yaar,"[ek, aisi, wife, to, main, bhi, deserve, karta..."
...,...,...,...
1235,Siapa yang akan Menjadi Raja Mata Uang Crypto ...,siapa yang akan menjadi raja mata uang crypto ...,"[siapa, yang, akan, menjadi, raja, mata, uang,..."
1236,Ktk Kampuni zenye Market Cap kubwa Duniani or ...,ktk kampuni zenye market cap kubwa duniani or ...,"[ktk, kampuni, zenye, market, cap, kubwa, duni..."
1237,meinmokhtar Hang keje tak Penuhkan ASB k sbb d...,meinmokhtar hang keje tak penuhkan asb k sbb d...,"[meinmokhtar, hang, keje, tak, penuhkan, asb, ..."
1238,Join disini Gabung belajar santuy bareng Crypt...,join disini gabung belajar santuy bareng crypt...,"[join, disini, gabung, belajar, santuy, bareng..."


In [15]:
print("Sebelum tokenizing : ", sample)

sample = tokenizer.tokenize(sample)
print("Setelah tokenizing : ", sample)

Sebelum tokenizing :  buat ngejelasin ke boomer yang bilang underwear eh underlyingnya bitcoin itu apa
Setelah tokenizing :  ['buat', 'ngejelasin', 'ke', 'boomer', 'yang', 'bilang', 'underwear', 'eh', 'underlyingnya', 'bitcoin', 'itu', 'apa']


In [16]:
#Normalizing
normalization_dict = pd.read_csv('../../kamuskatabaku.csv', names=['slang', 'formal'])
normalization_dict = dict(zip(normalization_dict.slang, normalization_dict.formal))
def normalize_text(tokens):
    normalized_tokens = [normalization_dict.get(token, token) for token in tokens]
    return normalized_tokens
df['normalized'] = df['tokens'].apply(normalize_text)
df

,full_text,case_folding,tokens,normalized
0,Chale Your man start dey do Crypto Football ay...,chale your man start dey do crypto football ay...,"[chale, your, man, start, dey, do, crypto, foo...","[chale, your, man, start, dey, di, crypto, foo..."
1,Bitcoin Orange OGsatoshis,bitcoin orange ogsatoshis,"[bitcoin, orange, ogsatoshis]","[bitcoin, orange, ogsatoshis]"
2,WahyuS Problem terbesar yg diselesaikan web ad...,wahyus problem terbesar yg diselesaikan web ad...,"[wahyus, problem, terbesar, yg, diselesaikan, ...","[wahyus, problem, terbesar, yang, diselesaikan..."
3,Buat ngejelasin ke boomer yang bilang Underwea...,buat ngejelasin ke boomer yang bilang underwea...,"[buat, ngejelasin, ke, boomer, yang, bilang, u...","[buat, menjelaskan, ke, boomer, yang, bilang, ..."
4,Ek aisi wife to main bhi deserve karta hu yaar,ek aisi wife to main bhi deserve karta hu yaar,"[ek, aisi, wife, to, main, bhi, deserve, karta...","[ek, aisi, wife, tapi, main, bhi, deserve, kar..."
...,...,...,...,...
1235,Siapa yang akan Menjadi Raja Mata Uang Crypto ...,siapa yang akan menjadi raja mata uang crypto ...,"[siapa, yang, akan, menjadi, raja, mata, uang,...","[siapa, yang, akan, menjadi, raja, mata, uang,..."
1236,Ktk Kampuni zenye Market Cap kubwa Duniani or ...,ktk kampuni zenye market cap kubwa duniani or ...,"[ktk, kampuni, zenye, market, cap, kubwa, duni...","[ktk, kampuni, zenye, market, cap, kubwa, duni..."
1237,meinmokhtar Hang keje tak Penuhkan ASB k sbb d...,meinmokhtar hang keje tak penuhkan asb k sbb d...,"[meinmokhtar, hang, keje, tak, penuhkan, asb, ...","[meinmokhtar, hang, keje, tak, penuhkan, asb, ..."
1238,Join disini Gabung belajar santuy bareng Crypt...,join disini gabung belajar santuy bareng crypt...,"[join, disini, gabung, belajar, santuy, bareng...","[join, disini, gabung, belajar, santuy, bareng..."


In [17]:
print("Sebelum normalization : ", sample)

sample = normalize_text(sample)
print("Setelah normalization : ", sample)

Sebelum normalization :  ['buat', 'ngejelasin', 'ke', 'boomer', 'yang', 'bilang', 'underwear', 'eh', 'underlyingnya', 'bitcoin', 'itu', 'apa']
Setelah normalization :  ['buat', 'menjelaskan', 'ke', 'boomer', 'yang', 'bilang', 'underwear', 'eh', 'underlyingnya', 'bitcoin', 'itu', 'apa']


In [18]:
#Stopword Removal
stopwords_id = set(stopwords.words('indonesian'))
stopwords_en = set(stopwords.words('english'))
stopwords = stopwords_id.union(stopwords_en)
def remove_stopwords(tokens):
    filtered_tokens = [token for token in tokens if token not in stopwords]
    return filtered_tokens
df['stopword_removal'] = df['normalized'].apply(remove_stopwords)
df

,full_text,case_folding,tokens,normalized,stopword_removal
0,Chale Your man start dey do Crypto Football ay...,chale your man start dey do crypto football ay...,"[chale, your, man, start, dey, do, crypto, foo...","[chale, your, man, start, dey, di, crypto, foo...","[chale, man, start, dey, crypto, football, ay,..."
1,Bitcoin Orange OGsatoshis,bitcoin orange ogsatoshis,"[bitcoin, orange, ogsatoshis]","[bitcoin, orange, ogsatoshis]","[bitcoin, orange, ogsatoshis]"
2,WahyuS Problem terbesar yg diselesaikan web ad...,wahyus problem terbesar yg diselesaikan web ad...,"[wahyus, problem, terbesar, yg, diselesaikan, ...","[wahyus, problem, terbesar, yang, diselesaikan...","[wahyus, problem, terbesar, diselesaikan, web,..."
3,Buat ngejelasin ke boomer yang bilang Underwea...,buat ngejelasin ke boomer yang bilang underwea...,"[buat, ngejelasin, ke, boomer, yang, bilang, u...","[buat, menjelaskan, ke, boomer, yang, bilang, ...","[boomer, bilang, underwear, eh, underlyingnya,..."
4,Ek aisi wife to main bhi deserve karta hu yaar,ek aisi wife to main bhi deserve karta hu yaar,"[ek, aisi, wife, to, main, bhi, deserve, karta...","[ek, aisi, wife, tapi, main, bhi, deserve, kar...","[ek, aisi, wife, main, bhi, deserve, karta, hu..."
...,...,...,...,...,...
1235,Siapa yang akan Menjadi Raja Mata Uang Crypto ...,siapa yang akan menjadi raja mata uang crypto ...,"[siapa, yang, akan, menjadi, raja, mata, uang,...","[siapa, yang, akan, menjadi, raja, mata, uang,...","[raja, mata, uang, crypto, bitcoin, postingan,..."
1236,Ktk Kampuni zenye Market Cap kubwa Duniani or ...,ktk kampuni zenye market cap kubwa duniani or ...,"[ktk, kampuni, zenye, market, cap, kubwa, duni...","[ktk, kampuni, zenye, market, cap, kubwa, duni...","[ktk, kampuni, zenye, market, cap, kubwa, duni..."
1237,meinmokhtar Hang keje tak Penuhkan ASB k sbb d...,meinmokhtar hang keje tak penuhkan asb k sbb d...,"[meinmokhtar, hang, keje, tak, penuhkan, asb, ...","[meinmokhtar, hang, keje, tak, penuhkan, asb, ...","[meinmokhtar, hang, keje, penuhkan, asb, dah, ..."
1238,Join disini Gabung belajar santuy bareng Crypt...,join disini gabung belajar santuy bareng crypt...,"[join, disini, gabung, belajar, santuy, bareng...","[join, disini, gabung, belajar, santuy, bareng...","[join, gabung, belajar, santuy, bareng, crypto..."


In [19]:
check = df.iloc[[3, 55, 56, 70]]
check

,full_text,case_folding,tokens,normalized,stopword_removal
3,Buat ngejelasin ke boomer yang bilang Underwea...,buat ngejelasin ke boomer yang bilang underwea...,"[buat, ngejelasin, ke, boomer, yang, bilang, u...","[buat, menjelaskan, ke, boomer, yang, bilang, ...","[boomer, bilang, underwear, eh, underlyingnya,..."
55,Makin dalem nih Bitcoin jatohnya Lanjut ke US ...,makin dalem nih bitcoin jatohnya lanjut ke us ...,"[makin, dalem, nih, bitcoin, jatohnya, lanjut,...","[makin, dalam, nih, bitcoin, jatohnya, lanjut,...","[nih, bitcoin, jatohnya, us, gan]"
56,BREAKING Bitcoin turun di,breaking bitcoin turun di,"[breaking, bitcoin, turun, di]","[breaking, bitcoin, turun, di]","[breaking, bitcoin, turun]"
70,Microstrategy sun kara siyan guda dubu akan fa...,microstrategy sun kara siyan guda dubu akan fa...,"[microstrategy, sun, kara, siyan, guda, dubu, ...","[microstrategy, sun, kara, siyan, guda, dubu, ...","[microstrategy, sun, kara, siyan, guda, dubu, ..."


In [20]:
#Stemming
factory = StemmerFactory()
stemmer = factory.create_stemmer()
def stemming(text):
    stemmed_text = [stemmer.stem(word) for word in text]
    return stemmed_text
df['stemming'] = df['stopword_removal'].apply(stemming)
df

,full_text,case_folding,tokens,normalized,stopword_removal,stemming
0,Chale Your man start dey do Crypto Football ay...,chale your man start dey do crypto football ay...,"[chale, your, man, start, dey, do, crypto, foo...","[chale, your, man, start, dey, di, crypto, foo...","[chale, man, start, dey, crypto, football, ay,...","[chale, man, start, dey, crypto, football, ay,..."
1,Bitcoin Orange OGsatoshis,bitcoin orange ogsatoshis,"[bitcoin, orange, ogsatoshis]","[bitcoin, orange, ogsatoshis]","[bitcoin, orange, ogsatoshis]","[bitcoin, orange, ogsatoshis]"
2,WahyuS Problem terbesar yg diselesaikan web ad...,wahyus problem terbesar yg diselesaikan web ad...,"[wahyus, problem, terbesar, yg, diselesaikan, ...","[wahyus, problem, terbesar, yang, diselesaikan...","[wahyus, problem, terbesar, diselesaikan, web,...","[wahyus, problem, besar, selesai, web, risiko,..."
3,Buat ngejelasin ke boomer yang bilang Underwea...,buat ngejelasin ke boomer yang bilang underwea...,"[buat, ngejelasin, ke, boomer, yang, bilang, u...","[buat, menjelaskan, ke, boomer, yang, bilang, ...","[boomer, bilang, underwear, eh, underlyingnya,...","[boomer, bilang, underwear, eh, underlyingnya,..."
4,Ek aisi wife to main bhi deserve karta hu yaar,ek aisi wife to main bhi deserve karta hu yaar,"[ek, aisi, wife, to, main, bhi, deserve, karta...","[ek, aisi, wife, tapi, main, bhi, deserve, kar...","[ek, aisi, wife, main, bhi, deserve, karta, hu...","[ek, aisi, wife, main, bhi, deserve, karta, hu..."
...,...,...,...,...,...,...
1235,Siapa yang akan Menjadi Raja Mata Uang Crypto ...,siapa yang akan menjadi raja mata uang crypto ...,"[siapa, yang, akan, menjadi, raja, mata, uang,...","[siapa, yang, akan, menjadi, raja, mata, uang,...","[raja, mata, uang, crypto, bitcoin, postingan,...","[raja, mata, uang, crypto, bitcoin, postingan,..."
1236,Ktk Kampuni zenye Market Cap kubwa Duniani or ...,ktk kampuni zenye market cap kubwa duniani or ...,"[ktk, kampuni, zenye, market, cap, kubwa, duni...","[ktk, kampuni, zenye, market, cap, kubwa, duni...","[ktk, kampuni, zenye, market, cap, kubwa, duni...","[ktk, kampuni, zenye, market, cap, kubwa, duni..."
1237,meinmokhtar Hang keje tak Penuhkan ASB k sbb d...,meinmokhtar hang keje tak penuhkan asb k sbb d...,"[meinmokhtar, hang, keje, tak, penuhkan, asb, ...","[meinmokhtar, hang, keje, tak, penuhkan, asb, ...","[meinmokhtar, hang, keje, penuhkan, asb, dah, ...","[meinmokhtar, hang, keje, penuh, asb, dah, n, ..."
1238,Join disini Gabung belajar santuy bareng Crypt...,join disini gabung belajar santuy bareng crypt...,"[join, disini, gabung, belajar, santuy, bareng...","[join, disini, gabung, belajar, santuy, bareng...","[join, gabung, belajar, santuy, bareng, crypto...","[join, gabung, ajar, santuy, bareng, cryptoriz..."
